In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PREFIX = "inertial-6286.188861:"

filename = "accel2.csv"

# ---------------------------------------------------------
# 1. Leer archivo desde DATA_START
# ---------------------------------------------------------

buff = []

with open(filename, "r") as file:
    data = False

    for line in file:
        line = line.strip()

        if line == "DATA_START":
            data = True
            continue

        if data:
            values = line.split(",")
            buff.append(values)

# ---------------------------------------------------------
# 2. Crear DataFrame
# ---------------------------------------------------------

header = [col.replace(PREFIX, "") for col in buff[0]]

df = pd.DataFrame(data=buff[1:], columns=header)

# Convertir columnas a numérico
for column in df.columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# ---------------------------------------------------------
# 3. Renombrar columnas largas a nombres simples
# ---------------------------------------------------------

rename_cols = {
    "scaledAccelX": "ax",
    "scaledAccelY": "ay",
    "scaledAccelZ": "az",

    "scaledGyroX": "wx",
    "scaledGyroY": "wy",
    "scaledGyroZ": "wz",

    "estLinearAccelX": "alx",
    "estLinearAccelX:valid": "alx_valid",

    "estLinearAccelY": "aly",
    "estLinearAccelY:valid": "aly_valid",

    "estLinearAccelZ": "alz",
    "estLinearAccelZ:valid": "alz_valid",
}

df = df.rename(columns=rename_cols)

# ---------------------------------------------------------
# 4. Crear tiempo relativo en segundos
# ---------------------------------------------------------
# El Time está en nanosegundos Unix

df["t"] = (df["Time"] - df["Time"].iloc[0]) / 1e9
# df["Time"] = (df["Time"] - df["Time"].iloc[0]) / 1e9

# ---------------------------------------------------------
# 5. Separar señales según el tipo de fila
# ---------------------------------------------------------

print(df.columns.tolist())

df_scaled = df.dropna(subset=["ax", "ay", "az", "wx", "wy", "wz"]).copy()

df_linear = df.dropna(subset=["alx", "aly", "alz"]).copy()

# Reiniciar índices
df_scaled = df_scaled.reset_index(drop=True)
df_linear = df_linear.reset_index(drop=True)

# ---------------------------------------------------------
# 6. Calcular dt real a partir del tiempo
# ---------------------------------------------------------

df_scaled["dt"] = df_scaled["t"].diff()
df_linear["dt"] = df_linear["t"].diff()

# Para la primera muestra, puedes usar el valor esperado
fs = 100
dt_nominal = 1 / fs

df_scaled.loc[0, "dt"] = dt_nominal
df_linear.loc[0, "dt"] = dt_nominal

print("DataFrame completo:")
# print(df.head())
df.head()

['Time', 'ax', 'ay', 'az', 'wx', 'wy', 'wz', 'alx', 'alx_valid', 'aly', 'aly_valid', 'alz', 'alz_valid', 't']
DataFrame completo:


,Time,ax,ay,az,wx,wy,wz,alx,alx_valid,aly,aly_valid,alz,alz_valid,t
0,1777333620713518592,0.007790,0.003237,-0.995851,0.000464,-0.000117,0.000160,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
1,1777333620716139776,NaN,NaN,NaN,NaN,NaN,NaN,0.006093,1.0,-0.011016,1.0,0.040337,1.0,0.002621
2,1777333620723537408,0.007763,0.003365,-0.995893,0.000378,-0.000197,-0.000019,NaN,NaN,NaN,NaN,NaN,NaN,0.010019
3,1777333620726153216,NaN,NaN,NaN,NaN,NaN,NaN,0.005853,1.0,-0.009720,1.0,0.039927,1.0,0.012635
4,1777333620733536512,0.007741,0.003499,-0.995834,0.000164,-0.000166,-0.000036,NaN,NaN,NaN,NaN,NaN,NaN,0.020018


In [18]:
df_scaled = df.dropna(subset=["ax", "ay", "az", "wx", "wy", "wz"]).copy()
df_linear = df.dropna(subset=["alx", "aly", "alz"]).copy()

df_join = pd.merge_asof(
    df_scaled.sort_values("Time"),
    df_linear[["Time", "alx", "alx_valid", "aly", "aly_valid", "alz", "alz_valid"]].sort_values("Time"),
    on="Time",
    direction="nearest",
    tolerance=5_000_000   # 5 ms en nanosegundos
)

df_join["t"] = (df_join["Time"] - df_join["Time"].iloc[0]) / 1e9

df_join.head()

,Time,ax,ay,az,wx,wy,wz,alx_x,alx_valid_x,aly_x,aly_valid_x,alz_x,alz_valid_x,t,alx_y,alx_valid_y,aly_y,aly_valid_y,alz_y,alz_valid_y
0,1777333620713518592,0.007790,0.003237,-0.995851,0.000464,-0.000117,0.000160,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.006093,1.0,-0.011016,1.0,0.040337,1.0
1,1777333620723537408,0.007763,0.003365,-0.995893,0.000378,-0.000197,-0.000019,NaN,NaN,NaN,NaN,NaN,NaN,0.010019,0.005853,1.0,-0.009720,1.0,0.039927,1.0
2,1777333620733536512,0.007741,0.003499,-0.995834,0.000164,-0.000166,-0.000036,NaN,NaN,NaN,NaN,NaN,NaN,0.020018,0.005645,1.0,-0.008349,1.0,0.040507,1.0
3,1777333620743516160,0.007636,0.003751,-0.995592,0.000197,0.000050,-0.000170,NaN,NaN,NaN,NaN,NaN,NaN,0.029998,0.004608,1.0,-0.005845,1.0,0.042888,1.0
4,1777333620753476096,0.007824,0.003529,-0.995580,0.000113,0.000067,-0.000255,NaN,NaN,NaN,NaN,NaN,NaN,0.039958,0.006437,1.0,-0.008004,1.0,0.043004,1.0
